In [0]:
%run /Workspace/de_shared/config



In [0]:
import time

def run_notebook(path, timeout=7200):
    print(f"Starting: {path}")
    start = time.time()
    dbutils.notebook.run(path, timeout)
    elapsed = round(time.time() - start, 1)
    print(f"Completed: {path} in {elapsed}s ✅")

try:
    print("=" * 50)
    print("PIPELINE START")
    print("=" * 50)

    run_notebook("/Workspace/de_shared/02_bronze_ingest")
    run_notebook("/Workspace/de_shared/03_silver_transform")
    run_notebook("/Workspace/de_shared/04_gold_scorecard")

    print("=" * 50)
    print("PIPELINE COMPLETE ✅")
    print("=" * 50)

except Exception as e:
    print("=" * 50)
    print(f"PIPELINE FAILED ❌: {str(e)}")
    print("=" * 50)
    raise

PIPELINE START
Starting: /Workspace/de_shared/02_bronze_ingest
Completed: /Workspace/de_shared/02_bronze_ingest in 192.3s ✅
Starting: /Workspace/de_shared/03_silver_transform
Completed: /Workspace/de_shared/03_silver_transform in 91.3s ✅
Starting: /Workspace/de_shared/04_gold_scorecard
Completed: /Workspace/de_shared/04_gold_scorecard in 71.4s ✅
PIPELINE COMPLETE ✅


In [0]:
from pyspark.sql.functions import col

def validate_pipeline():
    print("Running post-pipeline validation...")

    bronze = (spark.read.format("delta")
        .option("fs.s3a.access.key", AWS_ACCESS_KEY_ID)
        .option("fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
        .option("fs.s3a.endpoint", "s3.amazonaws.com")
        .load("s3a://medinsight-partd-yash/delta/bronze/partd_raw/"))

    silver = (spark.read.format("delta")
        .option("fs.s3a.access.key", AWS_ACCESS_KEY_ID)
        .option("fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
        .option("fs.s3a.endpoint", "s3.amazonaws.com")
        .load("s3a://medinsight-partd-yash/delta/silver/prescriber_claims/"))

    gold = (spark.read.format("delta")
        .option("fs.s3a.access.key", AWS_ACCESS_KEY_ID)
        .option("fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
        .option("fs.s3a.endpoint", "s3.amazonaws.com")
        .load("s3a://medinsight-partd-yash/delta/gold/territory_scorecard/"))

    print(f"Bronze rows : {bronze.count():,}")
    print(f"Silver rows : {silver.count():,}")
    print(f"Gold rows   : {gold.count():,}")

    assert bronze.count() == 25231862, "Bronze count mismatch"
    assert silver.count() == 23850038, "Silver count mismatch"
    assert gold.count()   == 23850038, "Gold count mismatch"
    assert gold.filter(col("prescriber_rank").isNull()).count() == 0, "Null ranks in Gold"

    print("All pipeline validations passed ✅")

validate_pipeline()

Running post-pipeline validation...
Bronze rows : 25,231,862
Silver rows : 23,850,038
Gold rows   : 23,850,038
All pipeline validations passed ✅
